# IOAI — 2025 Selection Camp Angry Birds (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, urllib.request
os.makedirs('data/features', exist_ok=True)
_base = 'https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/angrybirds-features'
for f in ['train.pt', 'val.pt', 'test.pt']:
    if not os.path.exists(f'data/features/{f}'):
        urllib.request.urlretrieve(f'{_base}/{f}', f'data/features/{f}')
print('데이터 준비:', 'data/features/train.pt' if os.path.exists('data/features/train.pt') else '실패')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Angry Birds — 스퓨리어스 상관 새 분류 (베이스라인)

> 데이터는 DGX에 **사전 준비**돼 있습니다 — `data/features/{train,val,test}.pt`(ResNet50 IMAGENET1K_V1 penultimate 2048-d 임베딩 + train/val 의 4-way 그룹 `y4`). 이미지 추출 없이 바로 분류기만 학습하면 됩니다(빠름).

**과제**: 새 종류(물새 0/뭍새 1)를 분류하되, 학습셋 뭍새에만 **빨간 사각형**(가짜 상관)이 섞여 있습니다. 지표 = 4개 그룹(새종류×배경) 중 **최저 정확도(worst-group acc)**(높을수록 좋음). held-out val 로 채점, 제출 `submission.csv`(datapointID, answer).

베이스라인: **평범한 로지스틱 회귀**(그룹 처리 없음). 다수 그룹에 치우쳐 **최저 그룹 정확도가 낮습니다**. (개선: 그룹 가중 샘플링 + **Group DRO** → 모범답안)

In [ ]:
import torch, numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
tr = torch.load('data/features/train.pt'); va = torch.load('data/features/val.pt')
ytr = (tr['y4'] // 2).numpy()      # 새 종류(0/1) = 4-way 그룹 // 2
print('train', tr['X'].shape, '| val', va['X'].shape)

In [ ]:
clf = LogisticRegression(max_iter=2000).fit(tr['X'].numpy(), ytr)
pred = clf.predict(va['X'].numpy())
y4 = va['y4'].numpy()
accs = [ (pred[y4==g] == (y4[y4==g]//2)).mean() for g in range(4) ]
print('per-group acc:', [round(float(a),4) for a in accs], '| worst:', round(float(min(accs)),4))
pd.DataFrame({'datapointID': va['id'], 'answer': pred}).to_csv('submission.csv', index=False)
print('wrote submission.csv', len(pred))

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)